In [7]:
from pathlib import Path
from tkinter import Tk, filedialog

import AERzip


def select_compressed_aedat_file():
    """Select a compressed AEDAT file and load/decompress it with AERzip."""
    initial_dir = Path.cwd().parent / "Compressed files"

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    selected = filedialog.askopenfilename(
        title="Select a compressed .aedat file",
        initialdir=str(initial_dir),
        filetypes=[
            ("AEDAT files", "*.aedat"),
            ("Compressed AEDAT files", "*.aedat.gz *.aedat.zip"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if not selected:
        raise FileNotFoundError("No file was selected.")

    selected_path = Path(selected)
    addresses, timestamps = AERzip.loadCompressedFile(str(selected_path), verbose=False)

    return addresses, timestamps

In [13]:
from collections import defaultdict, deque

THRESHOLD_SPIKES = 500  # Threshold for the number of spikes in a window to consider it "active".
WINDOW_TICKS = 10000000  # Window size equivalent to 0.1 seconds
TS_TICK_SECONDS = 1e-8  # 1 tick = 10 ns

addresses, timestamps = select_compressed_aedat_file()

# For each address, keep only spike times inside the current sliding window.
spike_times_by_addr = defaultdict(deque)
first_hit = None

for addr, ts in zip(addresses, timestamps):
    addr_i = int(addr)
    ts_i = int(ts)

    dq = spike_times_by_addr[addr_i]
    dq.append(ts_i)

    window_start = ts_i - WINDOW_TICKS
    while dq and dq[0] < window_start:
        dq.popleft()

    if len(dq) > THRESHOLD_SPIKES:
        first_hit = (ts_i, addr_i, len(dq))
        break

if first_hit is None:
    print(
        f"No window found where any address has more than {THRESHOLD_SPIKES} "
        f"spikes in {WINDOW_TICKS} ticks."
    )
else:
    ts, addr, spike_count = first_hit
    ts_seconds = ts * TS_TICK_SECONDS
    print(
        f"First timestamp above threshold in window: {ts_seconds:.3f} s "
    )

First timestamp above threshold in window: 4.095 s 
